# Prétraitement des fichiers MediaPipe Pose

Ce notebook applique, étape par étape, le pipeline de prétraitement défini dans `sources/preprocessing.py`.

**Philosophie :**

- Toute la logique de calcul est dans le module Python : ici, on se contente d'**orchestrer** et de **vérifier visuellement**.
- Une cellule = une étape conceptuelle.
- Les paramètres modifiables (filtre, chemins) sont rassemblés en haut du notebook.
- On fait des vérifications visuelles sur un seul participant, tout en construisant le jeu complet en arrière-plan.

## 1. Importation des bibliothèques et du module local

On importe le module `preprocessing` situé dans `sources/`. Pour que l'import fonctionne, on ajoute `sources/` au `sys.path`.

Le reste est la stack scientifique minimale utile pour les vérifications visuelles.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
SOURCES_DIR = PROJECT_DIR / "sources"
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
if str(SOURCES_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCES_DIR))

import preprocessing as pp

# Auto-reload preprocessing.py whenever it changes on disk
%load_ext autoreload
%autoreload 2

#pd.set_option("display.max_columns", 50)
#pd.set_option("display.width", 200)
#print(f"Project dir  : {PROJECT_DIR}")
#print(f"Notebooks    : {NOTEBOOKS_DIR}")
#print(f"Sources      : {SOURCES_DIR}")
#print(f"Data         : {DATA_DIR}")
#print(f"Results      : {RESULTS_DIR}")
#print("Module `preprocessing` imported successfully.")

## 2. Paramètres du pipeline

On centralise ici **tous les paramètres modifiables** du pipeline :

- les chemins (dossiers `data/`),
- les paramètres du filtre passe-bas Butterworth.

Pour tester d'autres valeurs de filtrage, il suffit d'éditer cette cellule et de relancer le notebook.

In [2]:
# --- Paths ---
DATA_DIR = PROJECT_DIR / "data"

# --- Filter parameters (edit here to test different values) ---
FPS = 30.0          # MediaPipe sampling rate, frames per second
CUTOFF_HZ = 6.0     # Low-pass cutoff frequency, Hz
FILTER_ORDER = 2    # Butterworth order (filtfilt effectively doubles it)

#print(f"DATA_DIR     : {DATA_DIR}")
print(f"FPS          : {FPS}")
print(f"CUTOFF_HZ    : {CUTOFF_HZ}")
print(f"FILTER_ORDER : {FILTER_ORDER}")

FPS          : 30.0
CUTOFF_HZ    : 6.0
FILTER_ORDER : 2


## 3. Repérage automatique des fichiers et identité des participants

On liste tous les fichiers CSV dans `data/P` et `data/SZ`, puis on en extrait l'identité (`group`, `participant`, `participant_id`).

**À vérifier visuellement :** 20 fichiers dans chaque groupe, 40 au total.

In [3]:
p_files, sz_files, all_files = pp.list_participant_files(DATA_DIR)
participants_df, participant_rows = pp.build_participant_table(all_files)

print(f"Found {len(p_files)} files in group P")
print(f"Found {len(sz_files)} files in group SZ")
print(f"Total             : {len(all_files)} files")
#participants_df.head(10)

Found 20 files in group P
Found 20 files in group SZ
Total             : 40 files


## 5. Construction du DataFrame long global

On construit le DataFrame long contenant **tous les participants**, filtré sur les landmarks d'intérêt (tête, épaules, poignets, hanches). On y ajoute les colonnes d'identité `group`, `participant`, `participant_id`.

In [4]:
df_all_long = pp.build_long_dataframe(participant_rows)

print(f"Global long shape   : {df_all_long.shape}")
print(f"Unique participants : {df_all_long['participant_id'].nunique()}")
print(f"Landmarks kept      : {sorted(df_all_long['landmark'].unique())}")
#df_all_long.head(10)

Global long shape   : (250650, 8)
Unique participants : 40
Landmarks kept      : ['left_hip', 'left_shoulder', 'left_wrist', 'nose', 'right_hip', 'right_shoulder', 'right_wrist']


## 6. Inversion de l'axe Y

MediaPipe place `y = 0` en haut de l'image. On applique `y = 1 - y` pour obtenir une convention biomécanique : un `y` plus grand = plus haut.

Après cette étape, un poignet levé aura un `y` supérieur à celui de l'épaule.

In [5]:
df_all_long = pp.invert_y_axis(df_all_long)

#print("y stats after inversion:")
#print(df_all_long["y"].describe())

## 7. Interpolation des coordonnées manquantes

Les valeurs `x` et `y` manquantes sont interpolées linéairement (participant par participant, landmark par landmark, triées par frame). Les NaN résiduels en début/fin de série sont comblés par forward-fill puis backward-fill.

**À vérifier visuellement :** le nombre de NaN sur `x` et `y` doit chuter à zéro (ou presque).

In [6]:
#before_interp_summary = pp.count_complete_landmark_signals(df_all_long)
#before_interp_summary

In [7]:
df_all_long = pp.interpolate_coordinates(df_all_long)

In [8]:
#after_interp_summary = pp.count_complete_landmark_signals(df_all_long)
#after_interp_summary

## 8. Passage au format large (wide)

Une ligne par `(participant, frame)`, une colonne par coordonnée de landmark (`nose_x`, `left_wrist_y`, etc.).

In [9]:
df_wide = pp.long_to_wide(df_all_long)

#print(f"Wide shape : {df_wide.shape}")
#print(f"Columns    : {list(df_wide.columns)}")
#df_wide.head()

In [10]:
coord_cols = [
    "nose_x", "nose_y",
    "left_shoulder_x", "left_shoulder_y",
    "right_shoulder_x", "right_shoulder_y",
    "left_wrist_x", "left_wrist_y",
    "right_wrist_x", "right_wrist_y",
    "left_hip_x", "left_hip_y",
    "right_hip_x", "right_hip_y",
]

coord_summary = pd.DataFrame([
    {
        "coordinate": col,
        "n_participants_total": df_wide["participant_id"].nunique(),
        "n_participants_usable": df_wide.groupby("participant_id")[col]
            .apply(lambda s: not s.isna().all())
            .sum()
    }
    for col in coord_cols
])

coord_summary

,coordinate,n_participants_total,n_participants_usable
0,nose_x,40,40
1,nose_y,40,40
2,left_shoulder_x,40,40
3,left_shoulder_y,40,40
4,right_shoulder_x,40,40
5,right_shoulder_y,40,40
6,left_wrist_x,40,37
7,left_wrist_y,40,37
8,right_wrist_x,40,36
9,right_wrist_y,40,36


## 9. Calcul des points dérivés et du centre d'image

On ajoute :

- `mid_shoulders` : milieu entre épaules,
- `mid_hips` : milieu entre hanches,
- `torso_center` : milieu entre `mid_shoulders` et `mid_hips`,
- `image_center` : point fixe à `(0.5, 0.5)`.

In [11]:
df_wide = pp.add_derived_points(df_wide)

#df_wide[[
#    "participant_id", "frame",
#    "mid_shoulders_x", "mid_shoulders_y",
#    "mid_hips_x", "mid_hips_y",
#    "torso_center_x", "torso_center_y",
#    "image_center_x", "image_center_y",
#]]

## 10. Largeur d'épaules et sa médiane par participant

La largeur d'épaules (distance euclidienne 2D entre les deux épaules) est calculée par frame, puis on en prend la médiane par participant. Cette médiane servira de facteur de normalisation robuste aux variations de taille et de distance caméra.

In [12]:
df_wide, median_shoulder_width = pp.compute_median_shoulder_width(df_wide)

#print("Median shoulder width per participant (first 10):")
#print(median_shoulder_width.head(10))
#print(f"\nRange: [{median_shoulder_width.min():.4f}, {median_shoulder_width.max():.4f}]")

## 11. Construction des signaux de mouvement relatifs

On construit les signaux relatifs (mains par rapport aux épaules, oscillation du torse, etc.) à partir des coordonnées **brutes**. Ils sont plus informatifs que les coordonnées absolues car ils suppriment la position globale du corps dans l'image.

In [13]:
df_wide = pp.build_movement_signals(df_wide)

movement_summary = pp.count_complete_movement_signals(df_wide)
movement_summary

# to inspect the movement signals for a specific participant, uncomment the lines below and set TEST_PID to a valid participant_id from df_wide
#TEST_PID = df_wide["participant_id"].iloc[0]
#df_wide.loc[
    #df_wide["participant_id"] == TEST_PID,
    #["participant_id", "frame"] + pp.MOVEMENT_SIGNALS,
#].head()

,movement,n_participants_total,n_participants_complete_100pct
0,head_horizontal,40,40
1,head_vertical,40,40
2,left_hand_horizontal,40,37
3,left_hand_vertical,40,37
4,right_hand_horizontal,40,36
5,right_hand_vertical,40,36
6,sideways,40,40
7,swaying,40,40


## 12. Normalisation des signaux de mouvement

Chaque signal de mouvement est divisé par la **médiane de largeur d'épaules** du participant correspondant. On crée explicitement des colonnes suffixées `_norm` pour **conserver les signaux bruts** côté à côte.

C'est la version normalisée qui sera filtrée à l'étape suivante.

In [14]:
df_wide = pp.normalize_movement_signals(df_wide, median_shoulder_width)

#df_wide.loc[
#    df_wide["participant_id"] == TEST_PID,
#    [
#        "participant_id", "frame",
#        "left_hand_horizontal", "left_hand_horizontal_norm",
#        "sideways", "sideways_norm",
#    ],
#].head()

## 13. Filtrage passe-bas Butterworth (zero-phase) sur les signaux normalisés

On applique un filtre Butterworth passe-bas via `filtfilt` (forward + backward, sans déphasage) sur les signaux **normalisés**. Le filtrage est fait **participant par participant** pour éviter la contamination inter-sujets.

Pour chaque `s_norm`, on crée une colonne `s_filt`.

Les paramètres `FPS`, `CUTOFF_HZ` et `FILTER_ORDER` viennent de la cellule de paramètres tout en haut du notebook.

In [15]:
df_wide = pp.filter_movement_signals(
    df_wide,
    fps=FPS,
    cutoff_hz=CUTOFF_HZ,
    order=FILTER_ORDER,
)

#print("Filtered signal columns created:")
#print([f"{s}_filt" for s in pp.MOVEMENT_SIGNALS])

## 14. Centrage des signaux filtrés

Chaque signal filtré est centré en soustrayant sa **moyenne par participant**, de sorte que tous les signaux oscillent autour de zéro.

**Vérification visuelle** : on trace pour un participant le signal brut, le signal filtré et le signal filtré + centré.

In [16]:
df_wide = pp.center_filtered_signals(df_wide)

#SIGNAL_TO_PLOT = "left_hand_vertical"
#sub = df_wide[df_wide["participant_id"] == TEST_PID].sort_values("frame")

#plt.figure(figsize=(11, 4))
#plt.plot(sub["frame"], sub[f"{SIGNAL_TO_PLOT}_norm"],          label="normalized (raw)", alpha=0.5)
#plt.plot(sub["frame"], sub[f"{SIGNAL_TO_PLOT}_filt"],          label="filtered",         linewidth=1, alpha=0.7)
#plt.plot(sub["frame"], sub[f"{SIGNAL_TO_PLOT}_filt_centered"], label="filtered+centered", linewidth=1)
#plt.axhline(0, color="k", linewidth=0.5)
#plt.title(f"Participant {TEST_PID} -- signal: {SIGNAL_TO_PLOT}")
#plt.xlabel("frame")
#plt.ylabel("value (shoulder-width units)")
#plt.legend()
#plt.tight_layout()
#plt.show()

## 15. Sauvegarde des deux fichiers CSV globaux

On écrit dans `data/` :

1. **`all_participants_raw_movements.csv`** : format large, tous participants, colonnes d'identité + signaux normalisés bruts (`*_norm`) + signaux filtrés (`*_filt`).
2. **`all_participants_preprocessed_movements.csv`** : format large, tous participants, colonnes d'identité + signaux filtrés et centrés. Les colonnes sont **renommées sans le suffixe** `_filt_centered` pour un usage plus simple dans les notebooks d'analyse ultérieurs.

In [17]:
raw_path, preproc_path = pp.save_outputs(df_wide, DATA_DIR)

#print(f"Saved raw movements          -> {raw_path}")
print(f"Saved preprocessed movements -> {preproc_path}")

Saved preprocessed movements -> c:\Users\factory\Desktop\pons.lea\data\all_participants_preprocessed_movements.csv
